In [1]:
import numpy as np
import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

In [2]:
df = pd.read_csv('../datasets/spam.csv', encoding='latin-1')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [3]:
df = df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'])
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [4]:
df.columns = ['label', 'message']
df.columns

Index(['label', 'message'], dtype='str')

In [5]:
df.isnull().sum()

label      0
message    0
dtype: int64

In [6]:
df['label'] = df['label'].map({'spam':1, 'ham':0})
df.head()

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df.drop_duplicates(inplace=True)
df.shape

(5169, 2)

In [8]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['message'] = df['message'].apply(clean_text)
df.head()

,label,message
0,0,go until jurong point crazy available only in ...
1,0,ok lar joking wif u oni
2,1,free entry in 2 a wkly comp to win fa cup fina...
3,0,u dun say so early hor u c already then say
4,0,nah i don t think he goes to usf he lives arou...


In [9]:
X = df['message']
y = df['label']

In [10]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(X).toarray()
X.shape

(5169, 3000)

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((4135, 3000), (1034, 3000))

In [12]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)
nb_pred

array([0, 0, 0, ..., 1, 0, 0], shape=(1034,))

In [13]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_pred

array([0, 0, 0, ..., 1, 0, 0], shape=(1034,))

In [14]:
cm=confusion_matrix(y_test, nb_pred)
ac=accuracy_score(y_test, nb_pred)
cr=classification_report(y_test, nb_pred)
print(cm)
print(ac)
print(cr)

[[889   0]
 [ 24 121]]
0.97678916827853
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       889
           1       1.00      0.83      0.91       145

    accuracy                           0.98      1034
   macro avg       0.99      0.92      0.95      1034
weighted avg       0.98      0.98      0.98      1034



In [15]:
cm=confusion_matrix(y_test, lr_pred)
ac=accuracy_score(y_test, lr_pred)
cr=classification_report(y_test, lr_pred)
print(cm)
print(ac)
print(cr)

[[885   4]
 [ 31 114]]
0.9661508704061895
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       889
           1       0.97      0.79      0.87       145

    accuracy                           0.97      1034
   macro avg       0.97      0.89      0.92      1034
weighted avg       0.97      0.97      0.96      1034



In [19]:
new_email = input("Enter email: ")

new_email = clean_text(new_email)

new_email_vector = tfidf.transform([new_email]).toarray()

prediction = nb_model.predict(new_email_vector)

prediction

array([1])